# 💥 Notebook 1: The Problem

Why are multi-step processes so hard?

## Learning Objectives

By the end of this notebook, you'll understand:
- What multi-step processes are
- Why they fail in unexpected ways
- The cost of partial failures

## 🛒 Real World Example: Order Fulfillment

In [ ]:
print("🛒 Order Fulfillment - Looks Simple, Right?")
print("=" * 60)
print("""
User clicks "Buy Now". What happens?

Step 1: Charge payment      → Talk to Stripe/PayPal
Step 2: Reserve inventory   → Update database
Step 3: Create shipping     → Talk to FedEx/UPS
Step 4: Send confirmation   → Send email
Step 5: Wait for pickup     → Human picks item
Step 6: Update tracking     → Webhook from carrier

Looks like a simple sequence... but it's NOT!
""")

In [ ]:
print("💥 What Can Go Wrong?")
print("=" * 60)
print("""
1. NETWORK FAILURES
   ─────────────────────────────────────────────────────────
   Your server ──► [✗ timeout] ──► Stripe
   
   Did the payment go through? You don't know!
   - Retry? Might double-charge!
   - Don't retry? Might lose the sale!

2. SERVER CRASHES
   ─────────────────────────────────────────────────────────
   Payment: ✅ Charged
   Inventory: ✅ Reserved
   Server: 💥 CRASH
   Shipping: ❓ Never created
   
   Customer is charged but never gets their item!

3. PARTIAL SUCCESS
   ─────────────────────────────────────────────────────────
   Payment: ✅ Charged
   Inventory: ❌ Out of stock!
   
   Now what? Need to refund, but that's another API call
   that might also fail!

4. LONG WAITS
   ─────────────────────────────────────────────────────────
   Waiting for human to pick item from warehouse...
   Could be 5 minutes or 5 hours.
   
   Can't keep HTTP connection open that long!
""")

## 🔥 Let's See It Fail

In [ ]:
import time
import random

class FlakyPaymentService:
    def charge(self, amount: float) -> dict:
        time.sleep(0.5)
        if random.random() < 0.3:
            raise TimeoutError("Payment gateway timeout")
        return {"transaction_id": "txn_123", "amount": amount}

class FlakyInventoryService:
    def __init__(self):
        self.stock = 5
    
    def reserve(self, item_id: str) -> dict:
        time.sleep(0.3)
        if self.stock <= 0:
            raise Exception("Out of stock!")
        if random.random() < 0.2:
            raise ConnectionError("Database connection lost")
        self.stock -= 1
        return {"reservation_id": f"res_{item_id}"}

class FlakyShippingService:
    def create_label(self, address: str) -> dict:
        time.sleep(0.4)
        if random.random() < 0.25:
            raise Exception("Shipping API error")
        return {"tracking_number": "1Z999AA10123456784"}

payment = FlakyPaymentService()
inventory = FlakyInventoryService()
shipping = FlakyShippingService()

print("✅ Flaky services initialized!")
print("   (30% payment timeout, 20% db error, 25% shipping error)")

In [ ]:
def process_order_naive(order_id: str, amount: float, item_id: str):
    print(f"\n📦 Processing order {order_id}...")
    
    print("   1️⃣ Charging payment...")
    payment_result = payment.charge(amount)
    print(f"      ✅ Payment: {payment_result['transaction_id']}")
    
    print("   2️⃣ Reserving inventory...")
    inventory_result = inventory.reserve(item_id)
    print(f"      ✅ Reserved: {inventory_result['reservation_id']}")
    
    print("   3️⃣ Creating shipping label...")
    shipping_result = shipping.create_label("123 Main St")
    print(f"      ✅ Tracking: {shipping_result['tracking_number']}")
    
    print(f"   ✅ Order {order_id} completed!")
    return {"status": "completed"}

print("🔥 Attempting 5 orders (expect some failures)...")
print("=" * 60)

results = {"success": 0, "failed": 0, "partial": []}

for i in range(5):
    try:
        process_order_naive(f"order_{i}", 99.99, f"item_{i}")
        results["success"] += 1
    except Exception as e:
        print(f"      ❌ FAILED: {e}")
        results["failed"] += 1
        results["partial"].append(f"order_{i}")

print(f"\n📊 Results: {results['success']} success, {results['failed']} failed")
if results["partial"]:
    print(f"   ⚠️ Partial failures: {results['partial']}")
    print("   These orders might have charged but not shipped!")

## 😱 The Partial Failure Problem

In [ ]:
print("😱 The Partial Failure Problem")
print("=" * 60)
print("""
When a step fails AFTER previous steps succeeded:

ORDER #42:
─────────────────────────────────────────────────────────────
Step 1: Payment    ✅ Charged $99.99 (transaction: txn_42)
Step 2: Inventory  ✅ Reserved item SKU-123
Step 3: Shipping   ❌ API timeout!

CURRENT STATE:
- Customer: Charged ✅
- Inventory: Locked ✅  
- Shipment: None ❌
- Customer Happy: ❌❌❌

WHAT DO WE DO NOW?
─────────────────────────────────────────────────────────────
Option A: Retry shipping
  - But what if it keeps failing?
  - How many retries? When to give up?

Option B: Roll back everything
  - Refund payment (another API call that might fail!)
  - Release inventory
  - What if refund fails? Retry? Loop forever?

Option C: Manual intervention
  - Alert on-call engineer at 3 AM
  - Engineer has to figure out state
  - Not scalable!
""")

In [ ]:
print("🤔 Why This Is Hard")
print("=" * 60)
print("""
DISTRIBUTED SYSTEMS ARE HARD BECAUSE:

1. NO GLOBAL STATE
   Each service has its own database.
   No single "transaction" spans all of them.
   
2. NO GUARANTEED DELIVERY
   Network can lose messages.
   Services can be down.
   Timeouts don't mean failure!
   
3. NO ORDERING GUARANTEES
   Messages can arrive out of order.
   Retries can cause duplicates.
   
4. NO INSTANTANEOUS COMMUNICATION
   Everything takes time.
   State can change while you're waiting.

DATABASES SOLVED THIS WITH ACID TRANSACTIONS...
BUT WE CAN'T USE THAT ACROSS SERVICES!
""")

## 🧪 Quick Quiz

1. **What's a partial failure?**

2. **Why can't we use database transactions across services?**

3. **What happens if payment succeeds but inventory fails?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Partial failure:")
print("   - Some steps succeeded, some failed")
print("   - System is in inconsistent state")
print("   - Hard to recover automatically")
print()
print("2. Why no cross-service transactions:")
print("   - Each service has its own database")
print("   - Network between services is unreliable")
print("   - Can't hold locks across network calls")
print()
print("3. Payment success + inventory fail:")
print("   - Customer is charged!")
print("   - Must refund (which might also fail)")
print("   - Need compensation logic")

## 📚 Summary

### The Problems

1. **Network failures** - Timeouts don't mean failure
2. **Server crashes** - Progress is lost
3. **Partial success** - Inconsistent state
4. **Long waits** - Can't hold connections

### What We Need

1. **State persistence** - Remember progress
2. **Automatic retry** - Keep trying on transient failures
3. **Compensation** - Undo completed steps on failure
4. **Crash recovery** - Continue from where we left off

### Next Up

In **Notebook 2**, we'll try the naive approach:
- Single server orchestration
- Manual state management
- Why it gets messy quickly